## <center> Sentiment Analysis on Twitter Sentiment140 Dataset using Logistic Regression </center>

In [ ]:
# installing kaggle library
! pip install kaggle

Upload Your Kaggle.json file

In [ ]:
mkdir -p ~/.kaggle && echo KGAT_7adae75f76e6e53c1beafb8041b4853f > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

Import Twitter Sentiment Dataset

In [ ]:
# API to fetch the dataset
!kaggle datasets download -d kazanova/sentiment140 -p /content/
!unzip /content/sentiment140.zip -d /content/

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
100% 80.9M/80.9M [00:00<00:00, 91.5MB/s]

Archive:  /content/sentiment140.zip
  inflating: /content/training.1600000.processed.noemoticon.csv  


In [ ]:
import os

# The dataset has already been unzipped into /content/training.1600000.processed.noemoticon.csv in the previous step.
# So, no further extraction is needed here.

# This cell can be removed or simplified if the file is directly available.
# For now, I'll ensure the next cell reads the CSV directly if the above unzipping creates it.
print('Assuming the dataset is unzipped and available at /content/training.1600000.processed.noemoticon.csv')

Assuming the dataset is unzipped and available at /content/training.1600000.processed.noemoticon.csv


Rename The dataset file

Importing the Dependencies

In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
print(stopwords.words('english'))       # Stopwords are words like "I", "me"
                                        # that do not add any influential meaning to the textual data
                                        # and can be removed from our textual input

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data Collection and Processing

In [ ]:
# loading the data from csv file to dataframe
twitter_data: pd.DataFrame = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1')

In [ ]:
# checking the number of and rows and columns
twitter_data.shape

(1599999, 6)

In [ ]:
# printing the first 5 rows of the datafream
twitter_data.head(5)

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [ ]:
# Naming the columns and reading the dataset again

column_names = ['Target','id','date','flag','user','text']
twitter_data: pd.DataFrame = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',names=column_names, encoding = 'ISO-8859-1')



In [ ]:
# checking the number of and rows and columns
twitter_data.shape

(1600000, 6)

In [ ]:
# printing the first 5 rows of the datafream
twitter_data.head(5)

,Target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [ ]:
#counting the nummber of missing values in the dataset
twitter_data.isnull().sum()

,0
Target,0
id,0
date,0
flag,0
user,0
text,0


In [ ]:
# checking the distribution of target column
twitter_data['Target'].value_counts()

,count
Target,
0,800000
4,800000


Convert 4  to 1 in the target column

In [ ]:
twitter_data.replace({'Target':{4:1}}, inplace = True)

In [ ]:
# checking the distribution of target column
twitter_data['Target'].value_counts()

,count
Target,
0,800000
1,800000


0-> Negative Tweet
1-> Positive Tweet

Stemming :
Stemming is the process of reducing a word  to its Root word
Example: actor,actress,acting --> act

In [ ]:
port_stem = PorterStemmer()

def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]', " ", content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if word not in stopwords.words('english')]
    stemmed_content = " ".join(stemmed_content)

    return stemmed_content

In [ ]:
twitter_data['stemmed_content'] = twitter_data['text'].apply(stemming)

In [ ]:
twitter_data.head(5)

,Target,id,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


In [ ]:
print(twitter_data['stemmed_content'])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemmed_content, Length: 1600000, dtype: object


In [ ]:
print(twitter_data['Target'])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: Target, Length: 1600000, dtype: int64


In [ ]:
#saparating the data and lable
x = twitter_data['stemmed_content'].values
y = twitter_data['Target'].values

In [ ]:
print(x)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [ ]:
print(y)

[0 0 0 ... 1 1 1]


Splitting the data to traning data and test data

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, stratify = y, random_state = 42)

In [ ]:
print(x.shape, x_train.shape, x_test.shape)

(1600000,) (1280000,) (320000,)


In [ ]:
print(x_train)

['paisleypaisley lol get idea far advanc even june yet need third knitter summer group'
 'worst headach ever'
 'ewaniesciuszko sad wont see miss alreadi yeah perfect come back th' ...
 'got home meet talk endlessli one coolest guy ever met smile'
 'bought chocol bar quot win free bar quot label win either'
 'misecia said hope dm email sunday']


In [ ]:
print(x_test)

['stm denali ye black red fav color realli want color def look awesom jare'
 'qu buy open hous weekend pm best valu one bedroom lic long island citi bd http tinyurl com pt nqd'
 'ginoandfran fran greet air okay hahahaha thank' ...
 'la brat follow also hope atleast get also wish get well soon'
 'feel like decent swell sinc last fall hope wave myrtl beach week either least golf'
 'relaxin busi day']


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, stratify = y, random_state = 42)
vactorizer = TfidfVectorizer()
vactorizer.fit(x_train)

x_train = vactorizer.transform(x_train)
x_test = vactorizer.transform(x_test)

In [ ]:
print(x_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9453607 stored elements and shape (1280000, 461280)>
  Coords	Values
  (0, 4832)	0.317074267861159
  (0, 124524)	0.18318401951949756
  (0, 128605)	0.22108856600702773
  (0, 146067)	0.12929728405657018
  (0, 154767)	0.26976607043258233
  (0, 175252)	0.224070805470346
  (0, 205794)	0.24140229063801746
  (0, 220296)	0.43015677907624866
  (0, 239679)	0.15130037108228483
  (0, 286478)	0.16123218610004272
  (0, 307108)	0.46206048815324474
  (0, 388138)	0.20555120011808467
  (0, 406297)	0.2978221095272138
  (0, 454381)	0.20169626473577715
  (1, 124611)	0.5113765148324884
  (1, 161801)	0.5778049407933611
  (1, 445870)	0.6361096685891185
  (2, 12436)	0.2529872032123258
  (2, 31063)	0.1936303169258752
  (2, 78861)	0.21039643874061958
  (2, 125319)	0.6383069130836649
  (2, 267649)	0.19309660201644555
  (2, 312657)	0.3154702974657607
  (2, 349409)	0.22232944888223494
  (2, 358186)	0.19837942712286838
  :	:
  (1279997, 80839)	0.408507509

In [ ]:
print(x_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2288367 stored elements and shape (320000, 461280)>
  Coords	Values
  (0, 28874)	0.1778395103911245
  (0, 43712)	0.23562815302828183
  (0, 78636)	0.5158100011206617
  (0, 96399)	0.255967788489452
  (0, 97585)	0.4019235611854435
  (0, 129417)	0.25650960779862714
  (0, 189057)	0.31324918577405797
  (0, 240451)	0.15341308097014625
  (0, 334643)	0.14719329779308424
  (0, 335577)	0.22602158147814247
  (0, 384697)	0.3281164007446601
  (0, 435956)	0.14183025329879742
  (0, 453357)	0.1781708363247895
  (1, 35118)	0.3128685670821343
  (1, 36669)	0.2729958846742327
  (1, 39445)	0.16867093960211466
  (1, 57115)	0.19043301054662504
  (1, 74274)	0.21148120876702692
  (1, 78790)	0.13386322067407883
  (1, 170374)	0.17525273735418329
  (1, 171245)	0.12468774856570086
  (1, 183279)	0.24586158827112847
  (1, 233854)	0.3852709938491561
  (1, 240223)	0.1674195650536303
  (1, 301683)	0.13212235134015302
  :	:
  (319997, 135536)	0.218099649775323

Treaning The Machine lerning Model

Logistic Regressshing

In [ ]:
model = LogisticRegression(max_iter=1000)

In [ ]:
model.fit(x_train, y_train)

LogisticRegression(max_iter=1000)

Model Evaluation

Accuricy Score

In [ ]:
# accuracy socre on the tranning data
X_train_prediction = model.predict(x_train)
training_data_accuracy = accuracy_score(y_train, X_train_prediction, )

In [ ]:
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.7999984375


In [ ]:
# accuracy socre on the tranning data
X_train_prediction = model.predict(x_test)
test_data_accuracy = accuracy_score(y_test, X_train_prediction, )

In [ ]:
print('Accuracy score on the tranning data :', test_data_accuracy)

Accuracy score on the tranning data : 0.776796875


Model accuracy = 77.67

Saving the trained model

In [ ]:
import pickle

In [ ]:
file_name = 'trained_model.sav'
pickle.dump(model, open(file_name, 'wb'))

Using the saved model for future prediction

In [ ]:
# Loading the saved model
loaded_model = pickle.load(open('/content/trained_model.sav', 'rb'))

In [ ]:
x_new = x_test[200]
y_new_true = y_test[200]
print(f"Actual sentiment (0=negative, 1=positive): {y_new_true}")

prediction = loaded_model.predict(x_new)
print(f"Predicted sentiment: {prediction[0]}")

if (prediction[0]==0):
  print('The news is Negative')
else:
  print('The news is Positive')

Actual sentiment (0=negative, 1=positive): 0
Predicted sentiment: 0
The news is Negative


In [ ]:
x_new = x_test[3]
y_new_true = y_test[3]
print(f"Actual sentiment (0=negative, 1=positive): {y_new_true}")

prediction = loaded_model.predict(x_new)
print(f"Predicted sentiment: {prediction[0]}")

if (prediction[0]==0):
  print('The news is Negative')
else:
  print('The news is Positive')

Actual sentiment (0=negative, 1=positive): 1
Predicted sentiment: 1
The news is Positive
